In [ ]:
import pytesseract
from PIL import Image
from pathlib import Path

# --- Settings ---
INPUT_DIR   = Path("raw_images")
OUTPUT_PATH = Path("session_07-05_2026.txt")
LANG        = "ita+eng+deu+fra"

TESS_CONFIG = (
    "--dpi 300 "
    "--psm 3 "
    "--oem 3 "
    "-c textord_max_noise_frac=0.7 "
    "-c preserve_interword_spaces=1"
)

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- Main ---
jpgs = sorted(INPUT_DIR.glob("*.jpg"))
if not jpgs:
    raise FileNotFoundError(f"No JPG files found in {INPUT_DIR.resolve()}")

print(f"Found {len(jpgs)} JPG file(s) in '{INPUT_DIR}'.")

texts = []
for i, jpg in enumerate(jpgs, start=1):
    print(f"  [{i}/{len(jpgs)}] {jpg.name}")
    img  = Image.open(jpg)
    text = pytesseract.image_to_string(img, lang=LANG, config=TESS_CONFIG)
    texts.append(f"--- {jpg.name} ---\n{text}")

OUTPUT_PATH.write_text("\n\n".join(texts), encoding="utf-8")
print(f"\nDone. Text saved to {OUTPUT_PATH}")

Accuracy check

In [18]:
import pytesseract
from PIL import Image, ImageOps
from pathlib import Path
import difflib
import re
import numpy as np

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

LANG        = "ita"
TESS_CONFIG = "--dpi 300 --psm 3 --oem 3"
IMAGE_PATH  = Path("raw_images") / "0271.jpg"
GT_PATH      = Path("page_0271.txt")
OCR4ALL_PATH = Path("page_0271_OCR4all.txt")

def preprocess(jpg_path):
    img = Image.open(jpg_path).convert("L")          # grayscale
    img = ImageOps.autocontrast(img)                 # stretch contrast
    threshold = np.array(img).mean()                 # Otsu-like threshold
    img = img.point(lambda x: 0 if x < threshold else 255, "1")  # binarize
    return img

# --- Normalize: join hyphenated line breaks, flatten to single block ---
def normalize(text):
    text = re.sub(r'-\n', '', text)    # join hyphenated line breaks
    text = re.sub(r'\n', ' ', text)    # replace remaining newlines with spaces
    text = re.sub(r' +', ' ', text)    # collapse multiple spaces
    return text.strip()

# --- OCR ---
ocr_text     = normalize(pytesseract.image_to_string(preprocess(IMAGE_PATH), lang=LANG, config=TESS_CONFIG))
gt_text      = normalize(GT_PATH.read_text(encoding="utf-8"))
ocr4all_text = normalize(OCR4ALL_PATH.read_text(encoding="utf-8"))

# --- Metrics ---
def cer(ref, hyp):
    ref, hyp = list(ref), list(hyp)
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1): d[i][0] = i
    for j in range(len(hyp) + 1): d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            d[i][j] = d[i-1][j-1] if ref[i-1] == hyp[j-1] else 1 + min(d[i-1][j], d[i][j-1], d[i-1][j-1])
    return d[len(ref)][len(hyp)] / max(len(ref), 1)

def wer(ref, hyp):
    ref, hyp = ref.split(), hyp.split()
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1): d[i][0] = i
    for j in range(len(hyp) + 1): d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            d[i][j] = d[i-1][j-1] if ref[i-1] == hyp[j-1] else 1 + min(d[i-1][j], d[i][j-1], d[i-1][j-1])
    return d[len(ref)][len(hyp)] / max(len(ref), 1)

print("                  CER        WER")
print(f"Tesseract     {cer(gt_text, ocr_text):>8.2%}   {wer(gt_text, ocr_text):>8.2%}")
print(f"OCR4all       {cer(gt_text, ocr4all_text):>8.2%}   {wer(gt_text, ocr4all_text):>8.2%}")

# --- Diff: Tesseract vs ground truth ---
print("\n--- Diff: ground truth vs Tesseract ---")
diff = difflib.unified_diff(
    gt_text.splitlines(), ocr_text.splitlines(),
    fromfile="ground_truth", tofile="tesseract", lineterm=""
)
print("\n".join(list(diff)[:80]))

# --- Diff: OCR4all vs ground truth ---
print("\n--- Diff: ground truth vs OCR4all ---")
diff = difflib.unified_diff(
    gt_text.splitlines(), ocr4all_text.splitlines(),
    fromfile="ground_truth", tofile="ocr4all", lineterm=""
)
print("\n".join(list(diff)[:80]))

                  CER        WER
Tesseract        1.31%      7.59%
OCR4all          5.67%     26.05%

--- Diff: ground truth vs Tesseract ---
--- ground_truth
+++ tesseract
@@ -1 +1 @@
-qualche ritardo sui termini fissati — sostiene che una riduzione dell'8% rappresenterebbe un sacrificio troppo forte per il Fondo, giacchè quest'ultimo potrebbe procurarsi i finanziamenti necessari sul mercato dei capitali a condizioni più favorevoli. Il Governo propone quindi la modifica del decreto di riscatto riducendo ulteriormente il coefficiente. In conformità a quanto avvenuto nel 55 al termine del coefficiente del 10%, potrebbe verificarsi che l'annuncio di una nuova agevolazione inducesse molti contribuenti a riscattare prima il debito d'imposta. Comunque, i motivi a favore e contro la riduzione del coefficiente di riscatto dovrebbero venir vagliati molto attentamente. Le mutate condizioni dei saggi di interesse nel mercato dei capitali non dovrebbero influire eccessivamente su tale decisione, 

From JPG to PDF with text layer

In [20]:
import subprocess
import tempfile
from pathlib import Path
from PIL import Image, ImageOps, ImageFilter
from pypdf import PdfWriter

TESSERACT    = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
INPUT_DIR    = Path("raw_images")
OUTPUT_PATH  = Path("session_07_05_2026_searchable.pdf")
LANG         = "ita"

# --- Image processing options ---
JPEG_QUALITY  = 85    # 1–95: lower = smaller file, less detail
CONVERT_GRAY  = True  # convert to grayscale (reduces file size, often better for OCR)
AUTO_CONTRAST = True  # stretch contrast to full range
SHARPEN       = False # apply sharpening filter (helps with blurry scans)
DENOISE       = False # apply smoothing to reduce scan noise (may blur thin strokes)
BINARIZE      = False # convert to pure black & white (smallest file, loses grey tones)
BINARIZE_THR  = 128   # threshold for binarization (0–255); lower = more black

def run(cmd, **kwargs):
    print("  $", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True, **kwargs)

def enhance_and_compress(jpg_path, out_path):
    img = Image.open(jpg_path)
    if CONVERT_GRAY:
        img = img.convert("L")
    if AUTO_CONTRAST:
        img = ImageOps.autocontrast(img)
    if DENOISE:
        img = img.filter(ImageFilter.SMOOTH_MORE)
    if SHARPEN:
        img = img.filter(ImageFilter.SHARPEN)
    if BINARIZE:
        img = img.convert("L").point(lambda x: 0 if x < BINARIZE_THR else 255, "1")
        img.save(out_path, format="PNG")  # PNG for binary (JPEG doesn't support 1-bit)
    else:
        img.save(out_path, format="JPEG", quality=JPEG_QUALITY, optimize=True)

jpgs = sorted(INPUT_DIR.glob("*.jpg"))
if not jpgs:
    raise FileNotFoundError(f"No JPG files found in {INPUT_DIR.resolve()}")

print(f"Found {len(jpgs)} JPG file(s).")

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)

    # 1. Enhance + compress each image, then OCR → per-page PDF
    print("Step 1: Enhancing and OCR-ing pages with Tesseract...")
    page_pdfs = []
    for i, jpg in enumerate(jpgs, start=1):
        print(f"  [{i}/{len(jpgs)}] {jpg.name}")
        ext = "png" if BINARIZE else "jpg"
        enhanced = tmp / f"enhanced_{i:04d}.{ext}"
        enhance_and_compress(jpg, enhanced)
        out_base = tmp / f"ocr_{i:04d}"
        run([TESSERACT, str(enhanced), str(out_base), "-l", LANG, "pdf"])
        page_pdfs.append(out_base.with_suffix(".pdf"))

    # 2. Merge and compress the final PDF
    print("Step 2: Merging and compressing...")
    writer = PdfWriter()
    for pdf_path in page_pdfs:
        writer.append(pdf_path)
    writer.compress_identical_objects(remove_identicals=True, remove_orphans=True)
    with open(OUTPUT_PATH, "wb") as f:
        writer.write(f)

size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024
print(f"\nDone. Searchable PDF saved to: {OUTPUT_PATH} ({size_mb:.1f} MB)")

Found 33 JPG file(s).
Step 1: Enhancing and OCR-ing pages with Tesseract...
  [1/33] 0258.jpg
  $ C:\Program Files\Tesseract-OCR\tesseract.exe C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\enhanced_0001.jpg C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\ocr_0001 -l ita pdf
  [2/33] 0259.jpg
  $ C:\Program Files\Tesseract-OCR\tesseract.exe C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\enhanced_0002.jpg C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\ocr_0002 -l ita pdf
  [3/33] 0260.jpg
  $ C:\Program Files\Tesseract-OCR\tesseract.exe C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\enhanced_0003.jpg C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\ocr_0003 -l ita pdf
  [4/33] 0261.jpg
  $ C:\Program Files\Tesseract-OCR\tesseract.exe C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\enhanced_0004.jpg C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\ocr_0004 -l ita pdf
  [5/33] 0262.jpg
  $ C:\Program Files\Tesseract-OCR\tesseract.exe C:\Users\paolo\AppData\Local\Temp\tmp5g36anux\enhanced_0005.jpg C:\Us

C:\Users\paolo\AppData\Local\Temp\ipykernel_2904\4288607251.py:67: DeprecationWarning: remove_identicals is deprecated and will be removed in pypdf 7.0.0. Use remove_duplicates instead.
  writer.compress_identical_objects(remove_identicals=True, remove_orphans=True)
C:\Users\paolo\AppData\Local\Temp\ipykernel_2904\4288607251.py:67: DeprecationWarning: remove_orphans is deprecated and will be removed in pypdf 7.0.0. Use remove_unreferenced instead.
  writer.compress_identical_objects(remove_identicals=True, remove_orphans=True)
